### TOTAL SALES: Total money collected from customers for all products sold.


In [0]:
SELECT 
  SUM(total_sales) AS total_money_collected 
FROM 
  cl_mp_de.`03_gold_fact`.fact_retail_analytics;

### PROFIT PER PRODUCT: Selling price minus the cost of the item.


In [0]:
SELECT
    dp.item_name,
    CASE 
      WHEN SUM(f.profit)>0
      THEN SUM(f.profit)
      ELSE 0 
    END AS total_profit
FROM cl_mp_de.`03_gold_fact`.fact_retail_analytics f
JOIN cl_mp_de.`03_gold_dim`.dim_product dp
  ON f.product_key = dp.product_key
GROUP BY dp.item_name
ORDER BY
  total_profit DESC;

### TOP SELLING CATEGORY: Which group (e.g. Electronics) brings the most money.


In [0]:
SELECT 
  dp.category_type AS top_selling_category,
  SUM(f.total_sales) AS total_sales
FROM
  cl_mp_de.`03_gold_fact`.fact_retail_analytics f
LEFT JOIN 
  cl_mp_de.`03_gold_dim`.dim_product dp
  ON f.product_key = dp.product_key
GROUP BY
  dp.category_type
ORDER BY
  total_sales DESC
LIMIT 1;

### BASKET SIZE: Average number of items bought per single transaction.

In [0]:
SELECT
    AVG(items_per_txn) AS avg_basket_size
FROM (
    SELECT
        transaction_id,
        SUM(quantity_sold) AS items_per_txn
    FROM cl_mp_de.`03_gold_fact`.fact_retail_analytics
    GROUP BY transaction_id
);

### RETURN RATE: Percentage of total sales that were returned by customers.


In [0]:
SELECT
    ROUND(
      SUM(return_flag) * 1.0 / COUNT(*), 
      2
    ) AS return_rate
FROM cl_mp_de.`03_gold_fact`.fact_retail_analytics;

### OUT-OF-STOCK COUNT: Count of unique products with zero units in stock.

In [0]:
SELECT 
  COUNT(DISTINCT(product_key)) AS out_of_stock_count
FROM 
  cl_mp_de.`03_gold_fact`.fact_retail_analytics
WHERE 
  stock_on_hand = 0;

### STORE PERFORMANCE: Ranking of the 5 stores based on daily sales volume.


In [0]:
WITH sales_agg AS (
    SELECT
        store_key,
        date_key,
        SUM(total_sales) AS total_sales
    FROM cl_mp_de.`03_gold_fact`.fact_retail_analytics
    GROUP BY store_key, date_key
),
ranked_sales AS(
  SELECT
      dd.full_date,
      sa.total_sales,
      sa.store_key,
      DENSE_RANK() OVER (
          PARTITION BY dd.date_key
          ORDER BY sa.total_sales DESC
      ) AS sales_rank
  FROM sales_agg sa
  JOIN cl_mp_de.`03_gold_dim`.dim_date dd
  ON sa.date_key = dd.date_key
)
SELECT *
FROM ranked_sales
WHERE sales_rank <= 5
ORDER BY full_date, sales_rank;

### DISCOUNT IMPACT: Money lost due to differences in marked vs. sold price.

In [0]:
SELECT
    SUM(discount_amount) AS total_discount_loss
FROM cl_mp_de.`03_gold_fact`.fact_retail_analytics;

### REPEAT CUSTOMER COUNT: Number of unique users with more than one purchase.


In [0]:
SELECT COUNT(*) AS repeat_customers
FROM (
  SELECT customer_key
  FROM cl_mp_de.`03_gold_fact`.fact_sales
  WHERE customer_key IS NOT NULL
  GROUP BY customer_key
  HAVING COUNT(transaction_id) > 1
);

### SLOW-MOVING INVENTORY: Products with zero sales recorded in the last 30 days.

In [0]:
SELECT dp.product_id, dp.item_name
FROM cl_mp_de.`03_gold_dim`.dim_product dp
LEFT JOIN (
    SELECT DISTINCT product_key
    FROM cl_mp_de.`03_gold_fact`.fact_retail_analytics
    WHERE date_key >= CAST(date_format(date_sub(current_date(), 30), 'yyyyMMdd') AS INT)
) recent_sales
ON dp.product_key = recent_sales.product_key
WHERE recent_sales.product_key IS NULL;